# Clustering: Welche Stationstypen gibt es wirklich?

**CRISP-DM-Block, Modeling-Phase (Block 05, Folie „Clustering findet Gruppen ganz ohne
vorgegebene Kategorien")**

## Worum es geht

In den ersten beiden Notebooks kannten wir die Zielgröße immer schon vorher — die
Fahrtdauer war eine Zahl in den Daten, "hat Meldung" ein Label aus einer anderen Tabelle.
Clustering ist etwas grundlegend anderes: **Wir geben dem Verfahren keine Zielgröße vor.**
Wir zeigen ihm nur Merkmale und lassen es selbst Gruppen finden.

## Was Clustering von den ersten beiden Verfahren unterscheidet

*"Beim Clustering geht es darum, bestehende Elemente anhand ihrer Merkmale in vorab
unbekannte Gruppen einzuteilen. Die Herausforderung liegt darin, die Gruppen so zu
bilden, dass die zugewiesenen Elemente hinsichtlich ihrer Merkmalsausprägungen möglichst
homogen sind. Zwischen den Gruppen sollten die Unterschiede hingegen möglichst groß
sein."* (Provost, F., Fawcett, T. (2015): Data Science für Unternehmen, S. 45 f.)

Regression und Klassifikation heißen **Supervised Learning**: Es gibt ein bekanntes
"richtiges" Ergebnis, aus dem das Modell lernt. Clustering ist **Unsupervised
Learning**: Es gibt keine bekannte Antwort — wir wollen ja gerade herausfinden, welche
Stationstypen es gibt, statt sie vorzugeben.

## Die Frage, die wir stellen

Bei der Konzeption von VeloCity haben wir zehn Stationen an unterschiedlichen Orten in
Würzburg angelegt — am Hauptbahnhof, an der Universität, an der Residenz. Es liegt nahe,
dass sich diese Orte im Nutzungsverhalten unterscheiden: Pendlerstationen morgens und
abends stark frequentiert, Freizeitstationen eher am Wochenende. **Aber woher wissen
wir das wirklich, ohne es vorher schon anzunehmen?** Genau das lässt sich mit
Clustering herausfinden, statt es zu unterstellen.

## Lernziele

1. Aus rohen Fahrtdaten ein **Stundenprofil** je Station bilden — Merkmale, die ein
   Nutzungsmuster beschreiben, nicht einen einzelnen Wert
2. Verstehen, warum Merkmale vor dem Clustering **standardisiert** werden müssen
3. Ein k-Means-Clustering durchführen und die Anzahl der Cluster begründet wählen
4. Die gefundenen Cluster gegen das eigene Geschäftsverständnis prüfen (Plausibilitätscheck
   statt Genauigkeits-Kennzahl — wie auf der Evaluation-Folie in Block 05 beschrieben)

## Schritt 1 — Bibliotheken importieren

Neu: `StandardScaler` zur Standardisierung und `KMeans` als Clustering-Verfahren.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 20)
print("Bibliotheken geladen.")

## Schritt 2 — Daten laden

**TODO:** Laden Sie `ausleihe.csv` (mit `startzeit` als Datum/Zeit geparst) und
`station.csv`.

In [ ]:
ausleihe = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/ausleihe.csv", parse_dates=["startzeit"])
station = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/station.csv")

print("Fahrten:", len(ausleihe))
print("Stationen:", len(station))
station[["station_id", "name"]]

## Schritt 3 — Das Stundenprofil bilden: von der Einzelfahrt zum Merkmalsvektor

Das ist der wichtigste gedankliche Schritt in diesem Notebook. Eine einzelne Fahrt sagt
uns wenig über eine Station. Aber **wie sich die Fahrten einer Station über den Tag
verteilen**, sagt sehr viel.

Wir bauen deshalb für jede Station ein **Stundenprofil**: den Anteil ihrer Fahrten, die
in jeder der 24 Tagesstunden beginnen. Zwei Stationen mit demselben Profil verhalten
sich ähnlich — auch wenn sie an ganz unterschiedlichen Orten liegen.

**TODO:**
1. Legen Sie eine Spalte `stunde` an (`.dt.hour` von `startzeit`).
2. Zählen Sie mit `pd.crosstab(...)` oder `.groupby([...]).size().unstack(...)`, wie
   viele Fahrten je Station (`start_station_id`) in jeder Stunde beginnen. Das Ergebnis
   soll eine Tabelle sein: Zeilen = Stationen, Spalten = Stunden 0–23.
3. Wandeln Sie die absoluten Zahlen in **Anteile** um: teilen Sie jede Zeile durch ihre
   Summe (`.div(zeilensumme, axis=0)`) — sonst würden stark frequentierte Stationen
   allein wegen ihres Gesamtvolumens anders geclustert als schwach frequentierte,
   obwohl uns nur das *Muster* interessiert, nicht die Menge.

In [ ]:
ausleihe["stunde"] = ...  # TODO

profil_absolut = pd.crosstab(ausleihe["start_station_id"], ausleihe["stunde"])
profil = ...  # TODO: profil_absolut in Zeilenanteile umwandeln (durch die Zeilensumme teilen)

profil.round(2)

**Kurzer Blick vorab:** Zeichnen Sie das Profil zweier Stationen Ihrer Wahl übereinander
(`profil.loc[station_id].plot()`), zum Beispiel Station 1 (Hauptbahnhof) und Station 2
(Residenz). Sehen Sie schon mit bloßem Auge einen Unterschied?

In [ ]:
profil.loc[1].plot(label="Station 1", figsize=(9, 4))
profil.loc[2].plot(label="Station 2")
plt.xlabel("Stunde")
plt.ylabel("Anteil der Tagesfahrten")
plt.legend()
plt.title("Stundenprofil zweier Stationen im Vergleich")
plt.show()

## Schritt 4 — Merkmale standardisieren

k-Means misst Ähnlichkeit über den **Abstand** zwischen Merkmalsvektoren. Da unsere
24 Stundenanteile schon alle auf derselben Skala liegen (Anteile zwischen 0 und 1),
ist der Standardisierungsschritt hier weniger dramatisch als bei sehr unterschiedlich
skalierten Merkmalen (z. B. Kilometer neben Minuten) — trotzdem ist es gute Praxis,
ihn nie wegzulassen: `StandardScaler` zentriert jede Spalte auf Mittelwert 0 und
Standardabweichung 1, damit keine einzelne Stunde durch zufällig höhere Varianz das
Ergebnis dominiert.

**TODO:** Erzeugen Sie einen `StandardScaler()` und wenden Sie `.fit_transform(profil)`
darauf an.

In [ ]:
scaler = ...  # TODO: StandardScaler()
profil_skaliert = ...  # TODO: scaler.fit_transform(profil)

print(profil_skaliert.shape)

## Schritt 5 — k-Means clustern

k-Means teilt die Stationen in *k* Gruppen ein, indem es Gruppenmittelpunkte
("Zentroide") so lange verschiebt, bis jede Station möglichst nah an ihrem eigenen
Zentroid liegt. Die Anzahl *k* muss vorab festgelegt werden — anders als bei
Klassifikation gibt es hier keine "richtige" Zahl, die aus den Daten abzulesen wäre.

Aus der Konzeption kennen wir vier gedachte Stationstypen (Pendler, Uni, Freizeit,
Misch) — wir probieren deshalb `k=3` und `k=4` aus und vergleichen.

**TODO:** Trainieren Sie `KMeans(n_clusters=3, random_state=42, n_init=10)` und
`KMeans(n_clusters=4, random_state=42, n_init=10)` auf `profil_skaliert`. Speichern Sie
die Cluster-Zuordnung (`.labels_`) jeweils als neue Spalte in `station`.

In [ ]:
kmeans3 = ...  # TODO: KMeans(n_clusters=3, random_state=42, n_init=10).fit(profil_skaliert)
kmeans4 = ...  # TODO: KMeans(n_clusters=4, random_state=42, n_init=10).fit(profil_skaliert)

station = station.set_index("station_id")
station["cluster_k3"] = kmeans3.labels_
station["cluster_k4"] = kmeans4.labels_
station[["name", "cluster_k3", "cluster_k4"]].sort_values("cluster_k4")

## Schritt 6 — Plausibilitätscheck: ergeben die Cluster inhaltlich Sinn?

Auf der Evaluation-Folie in Block 05 steht dazu ausdrücklich: *"Clustering: kein
Genauigkeitsmaß, sondern Plausibilitätscheck gegen das Geschäftsverständnis."* Es gibt
hier keine MAE- oder Confusion-Matrix-Zahl, die uns sagt, ob das Ergebnis "richtig" ist
— wir müssen selbst beurteilen, ob die gefundenen Gruppen Sinn ergeben.

**TODO:** Schauen Sie sich die Tabelle aus Schritt 5 an und ordnen Sie die Cluster in
Worten zu:
1. Welche Stationen landen bei `k=4` zusammen in einer Gruppe?
2. Passt das zu dem, was Sie über Hauptbahnhof, Residenz, Sanderring/Hubland (Uni) und
   Zellerau/Grombühl (Wohngebiete) wissen — ohne dass wir dem Algorithmus diese
   Kategorien je genannt hätten?
3. Was ändert sich zwischen `k=3` und `k=4`? Werden zwei inhaltlich unterschiedliche
   Gruppen bei `k=3` zusammengelegt?

*Ihre Antwort hier …*

## Schritt 7 — Die Stundenprofile je Cluster visualisieren

Ein Bild sagt hier mehr als die Tabelle. Zeichnen wir für `k=4` das durchschnittliche
Stundenprofil jedes Clusters — dann wird auf einen Blick sichtbar, ob die Cluster
tatsächlich unterschiedliche Tagesrhythmen abbilden.

**TODO:** Gruppieren Sie `profil` (die *unskalierten* Anteile, zur besseren
Lesbarkeit) nach `station["cluster_k4"]` und bilden Sie je Cluster den Mittelwert über
alle 24 Stunden. Zeichnen Sie das Ergebnis.

In [ ]:
profil_mit_cluster = profil.copy()
profil_mit_cluster["cluster"] = station["cluster_k4"].values

mittelwert_je_cluster = ...  # TODO: profil_mit_cluster.groupby("cluster").mean()

mittelwert_je_cluster.T.plot(figsize=(10, 5), marker="o")
plt.xlabel("Stunde")
plt.ylabel("⌀ Anteil der Tagesfahrten")
plt.title("Durchschnittliches Stundenprofil je Cluster")
plt.legend(title="Cluster")
plt.show()

## Zusammenfassung

In diesem Notebook haben Sie:
- aus Einzelfahrten ein aggregiertes Merkmal je Station gebildet (ein 24-dimensionales
  Stundenprofil statt eines einzelnen Werts),
- verstanden, warum Merkmale vor dem Clustering standardisiert werden,
- k-Means mit unterschiedlicher Clusterzahl angewendet und verglichen,
- die gefundenen Gruppen gegen das eigene Geschäftsverständnis geprüft — ganz ohne
  vorgegebene Labels sind dieselben Stationstypen sichtbar geworden, die wir bei der
  Konzeption von VeloCity angelegt hatten.

**Weiter geht's mit Notebook 4 — Zeitreihe:** Dort sagen wir keine Gruppe mehr voraus,
sondern einen zukünftigen Wert: wie viele Fahrten kommen morgen?